In [21]:
%pip install -q optuna lightgbm xgboost catboost scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import pandas as pd
import numpy as np
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
import xgboost as xgb
import catboost
import joblib
import time

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
CPU_THREADS = 4

MODEL_NAMES = [
    "LightGBM",
    "CatBoost",
    "XGBoost",
    "ElasticNet"
]

TARGETS = {
    "Income": "target_income",
    "Expense": "target_expense",
    "Balance": "target_balance"
}

# Balanced Optuna tuning budget
N_TRIALS = {
    "LightGBM": 20,
    "CatBoost": 15,
    "XGBoost": 20,
    "ElasticNet": 15
}

print("FINOVA AI - Forecasting Configuration Ready")

FINOVA AI - Forecasting Configuration Ready


In [ ]:
# Reload datasets without treating first row as header
df1 = pd.read_csv("fin_trans.tsv", sep="\t", header=None, low_memory=False)
df2 = pd.read_csv("fin_account.tsv", sep="\t", header=None, low_memory=False)
df3 = pd.read_csv("fin_order.tsv", sep="\t", header=None, low_memory=False)

print("===== DATASET SHAPES =====")
print("Transaction:", df1.shape)
print("Account:", df2.shape)
print("Order:", df3.shape)

print("\n===== TRANSACTION SAMPLE =====")
print(df1.head(3))

print("\n===== ACCOUNT SAMPLE =====")
print(df2.head(3))

print("\n===== ORDER SAMPLE =====")
print(df3.head(3))

===== DATASET SHAPES =====
Transaction: (1056320, 10)
Account: (4500, 4)
Order: (6471, 6)

===== TRANSACTION SAMPLE =====
   0  1           2      3        4  5    6   7    8           9
0  1  1  2015-03-24  100.0   100.00  C  CIC      NaN         NaN
1  5  1  2015-04-13  367.9   467.90  C  COB       AB  41403269.0
2  6  1  2015-05-13  367.9  2097.72  C  COB       AB  41403269.0

===== ACCOUNT SAMPLE =====
   0   1           2  3
0  1  18  2015-03-24  M
1  2   1  2013-02-26  M
2  3   5  2017-07-07  M

===== ORDER SAMPLE =====
       0  1   2         3       4   5
0  29401  1  YZ  87144583  245.20  HH
1  29402  2  ST  89597016  337.27  LO
2  29403  2  QR  13943797  726.60  HH


## Assign Proper Column Names

In [6]:
# Transaction Dataset
df1.columns = [
    "trans_id",
    "account_id",
    "trans_date",
    "amount",
    "balance",
    "trans_type",
    "operation",
    "category",
    "other_bank_id",
    "other_account_id"
]

# Account Dataset
df2.columns = [
    "account_id",
    "district_id",
    "create_date",
    "frequency"
]

# Order Dataset
df3.columns = [
    "order_id",
    "account_id",
    "bank_to",
    "account_to",
    "amount",
    "category"
]

# Convert dates
df1["trans_date"] = pd.to_datetime(df1["trans_date"])
df2["create_date"] = pd.to_datetime(df2["create_date"])

# Check results
print("===== TRANSACTIONS =====")
print(df1.head(3))

print("\n===== ACCOUNTS =====")
print(df2.head(3))

print("\n===== ORDERS =====")
print(df3.head(3))

===== TRANSACTIONS =====
   trans_id  account_id trans_date  amount  balance trans_type operation  \
0         1           1 2015-03-24   100.0   100.00          C       CIC   
1         5           1 2015-04-13   367.9   467.90          C       COB   
2         6           1 2015-05-13   367.9  2097.72          C       COB   

  category other_bank_id  other_account_id  
0                    NaN               NaN  
1                     AB        41403269.0  
2                     AB        41403269.0  

===== ACCOUNTS =====
   account_id  district_id create_date frequency
0           1           18  2015-03-24         M
1           2            1  2013-02-26         M
2           3            5  2017-07-07         M

===== ORDERS =====
   order_id  account_id bank_to  account_to  amount category
0     29401           1      YZ    87144583  245.20       HH
1     29402           2      ST    89597016  337.27       LO
2     29403           2      QR    13943797  726.60       HH


## Full Dataset Health Check

In [7]:
datasets = {
    "TRANSACTIONS": df1,
    "ACCOUNTS": df2,
    "ORDERS": df3
}

for name, df in datasets.items():

    print(f"\n{'='*15} {name} {'='*15}")

    print("\nDataset Shape:")
    print(df.shape)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)

    print("\nUnique Accounts:")
    print(df["account_id"].nunique())


print("\n========== TRANSACTION ANALYSIS ==========")

print("\nTransaction Date Range:")
print("Start:", df1["trans_date"].min())
print("End:", df1["trans_date"].max())

print("\nTransaction Types:")
print(df1["trans_type"].value_counts(dropna=False))

print("\nNegative Amounts:")
print((df1["amount"] < 0).sum())

print("\nZero Amounts:")
print((df1["amount"] == 0).sum())

print("\nNegative Balances:")
print((df1["balance"] < 0).sum())


print("\n========== ACCOUNT INTEGRITY ==========")

print("Transaction Accounts Missing in Account Table:")
print(
    (~df1["account_id"].isin(df2["account_id"])).sum()
)

print("\nOrder Accounts Missing in Account Table:")
print(
    (~df3["account_id"].isin(df2["account_id"])).sum()
)


=============== TRANSACTIONS ===============

Dataset Shape:
(1056320, 10)

Missing Values:
trans_id                 0
account_id               0
trans_date               0
amount                   0
balance                  0
trans_type               0
operation                0
category                10
other_bank_id       760931
other_account_id    760931
dtype: int64

Duplicate Rows:
0

Data Types:
trans_id                     int64
account_id                   int64
trans_date          datetime64[ns]
amount                     float64
balance                    float64
trans_type                  object
operation                   object
category                    object
other_bank_id               object
other_account_id           float64
dtype: object

Unique Accounts:
4500

=============== ACCOUNTS ===============

Dataset Shape:
(4500, 4)

Missing Values:
account_id     0
district_id    0
create_date    0
frequency      0
dtype: int64

Duplicate Rows:
0

Data Types:
account

## Financial Transaction Validation

In [8]:
print("===== AMOUNT BY TRANSACTION TYPE =====")

print(
    df1.groupby("trans_type")["amount"].agg(
        ["count", "min", "max", "mean", "sum"]
    )
)

print("\n===== AMOUNT SIGN DISTRIBUTION =====")

df1["amount_sign"] = np.select(
    [
        df1["amount"] < 0,
        df1["amount"] == 0,
        df1["amount"] > 0
    ],
    ["Negative", "Zero", "Positive"],
    default="Unknown"
)

print(
    pd.crosstab(
        df1["trans_type"],
        df1["amount_sign"]
    )
)

print("\n===== TRANSACTION CATEGORY =====")
print(df1["category"].value_counts(dropna=False).head(20))

print("\n===== OPERATION TYPES =====")
print(df1["operation"].value_counts(dropna=False))

print("\n===== UNIQUE TRANSACTION IDs =====")
print(df1["trans_id"].nunique())

print("\n===== AMOUNT SUMMARY =====")
print(df1["amount"].describe())

===== AMOUNT BY TRANSACTION TYPE =====
             count     min     max         mean           sum
trans_type                                                   
C           405083     0.0  7481.2   796.743488  3.227472e+08
D           634571 -8740.0     0.0  -444.665353 -2.821717e+08
P            16666 -2499.7  -200.1 -1251.672615 -2.086038e+07

===== AMOUNT SIGN DISTRIBUTION =====
amount_sign  Negative  Positive  Zero
trans_type                           
C                   0    405079     4
D              634561         0    10
P               16666         0     0

===== TRANSACTION CATEGORY =====
category
       535304
IC     183114
ST     155832
HH     118065
PE      30338
IN      18500
LO      13580
IO       1577
NaN        10
Name: count, dtype: int64

===== OPERATION TYPES =====
operation
WIC    434918
ROB    208283
       183114
CIC    156743
COB     65226
CCW      8036
Name: count, dtype: int64

===== UNIQUE TRANSACTION IDs =====
1056320

===== AMOUNT SUMMARY =====
count  

## Account History & Forecasting Readiness

In [9]:
# Account-level transaction history

account_history = df1.groupby("account_id").agg(
    total_transactions=("trans_id", "count"),
    first_transaction=("trans_date", "min"),
    last_transaction=("trans_date", "max"),
    total_cash_flow=("amount", "sum")
)

# Calculate history duration
account_history["history_days"] = (
    account_history["last_transaction"]
    - account_history["first_transaction"]
).dt.days

account_history["history_months"] = (
    account_history["history_days"] / 30.44
).round(1)

print("===== ACCOUNT HISTORY SUMMARY =====")
print(account_history.describe())

print("\n===== FORECASTING READINESS =====")

print("Total Accounts:", len(account_history))

print(
    "Accounts with 6+ Months:",
    (account_history["history_months"] >= 6).sum()
)

print(
    "Accounts with 12+ Months:",
    (account_history["history_months"] >= 12).sum()
)

print(
    "Accounts with 24+ Months:",
    (account_history["history_months"] >= 24).sum()
)

print(
    "Accounts with 36+ Months:",
    (account_history["history_months"] >= 36).sum()
)

print("\n===== TRANSACTIONS PER ACCOUNT =====")
print(account_history["total_transactions"].describe())

print("\n===== HISTORY MONTHS =====")
print(account_history["history_months"].describe())

===== ACCOUNT HISTORY SUMMARY =====
       total_transactions           first_transaction  \
count         4500.000000                        4500   
mean           234.737778  2015-08-08 01:38:52.800000   
min              9.000000         2013-01-01 00:00:00   
25%            133.000000         2013-12-27 00:00:00   
50%            208.000000         2016-01-02 00:00:00   
75%            330.000000         2016-11-01 00:00:00   
max            675.000000         2017-12-29 00:00:00   
std            126.849443                         NaN   

                    last_transaction  total_cash_flow  history_days  \
count                           4500      4500.000000   4500.000000   
mean   2018-12-28 09:37:16.800000256      4381.139749   1238.332222   
min              2016-08-25 00:00:00     -2582.100000    330.000000   
25%              2018-12-31 00:00:00      2307.510000    786.000000   
50%              2018-12-31 00:00:00      3849.485000   1093.000000   
75%              2018-12

## Monthly Activity Validation

In [10]:
# Create monthly period
df1["year_month"] = df1["trans_date"].dt.to_period("M")

# Number of active months per account
monthly_activity = (
    df1.groupby("account_id")["year_month"]
    .nunique()
    .reset_index(name="active_months")
)

# Calendar months between first and last transaction
account_history["calendar_months"] = (
    (
        account_history["last_transaction"].dt.year
        - account_history["first_transaction"].dt.year
    ) * 12
    +
    (
        account_history["last_transaction"].dt.month
        - account_history["first_transaction"].dt.month
    )
    + 1
)

# Combine results
monthly_activity = monthly_activity.merge(
    account_history[["calendar_months"]],
    left_on="account_id",
    right_index=True
)

# Calculate inactive months
monthly_activity["inactive_months"] = (
    monthly_activity["calendar_months"]
    - monthly_activity["active_months"]
)

# Activity ratio
monthly_activity["activity_ratio"] = (
    monthly_activity["active_months"]
    / monthly_activity["calendar_months"]
)

print("===== MONTHLY ACTIVITY SUMMARY =====")
print(monthly_activity.describe())

print("\n===== ACCOUNT ACTIVITY LEVELS =====")

print(
    "Accounts with 90%+ Active Months:",
    (monthly_activity["activity_ratio"] >= 0.90).sum()
)

print(
    "Accounts with 75%+ Active Months:",
    (monthly_activity["activity_ratio"] >= 0.75).sum()
)

print(
    "Accounts with 50%+ Active Months:",
    (monthly_activity["activity_ratio"] >= 0.50).sum()
)

print("\n===== INACTIVE MONTHS =====")
print(monthly_activity["inactive_months"].describe())

print("\n===== ACCOUNT ELIGIBILITY =====")

eligible = monthly_activity[
    (monthly_activity["calendar_months"] >= 12)
    & (monthly_activity["activity_ratio"] >= 0.75)
]

print("Potential Eligible Accounts:", len(eligible))

===== MONTHLY ACTIVITY SUMMARY =====
         account_id  active_months  calendar_months  inactive_months  \
count   4500.000000    4500.000000      4500.000000      4500.000000   
mean    2786.067556      41.123778        41.183556         0.059778   
std     2313.811984      18.153406        18.131747         0.612335   
min        1.000000       9.000000        12.000000         0.000000   
25%     1182.750000      26.000000        26.000000         0.000000   
50%     2368.000000      36.000000        36.000000         0.000000   
75%     3552.250000      61.000000        61.000000         0.000000   
max    11382.000000      72.000000        72.000000        16.000000   

       activity_ratio  
count     4500.000000  
mean         0.998159  
std          0.018736  
min          0.500000  
25%          1.000000  
50%          1.000000  
75%          1.000000  
max          1.000000  

===== ACCOUNT ACTIVITY LEVELS =====
Accounts with 90%+ Active Months: 4479
Accounts with 75%+ Act

## Monthly Financial Aggregation

In [11]:
# Work with eligible accounts only
eligible_ids = eligible["account_id"]

df_monthly = df1[
    df1["account_id"].isin(eligible_ids)
].copy()

# Create financial features
df_monthly["income"] = df_monthly["amount"].clip(lower=0)

df_monthly["expense"] = (
    -df_monthly["amount"].clip(upper=0)
)

# Sort transactions
df_monthly = df_monthly.sort_values(
    ["account_id", "trans_date", "trans_id"]
)

# Aggregate monthly financial data
monthly_finance = (
    df_monthly.groupby(
        ["account_id", "year_month"]
    )
    .agg(
        income=("income", "sum"),
        expense=("expense", "sum"),
        transaction_count=("trans_id", "count"),
        month_end_balance=("balance", "last")
    )
    .reset_index()
)

# Net cash flow
monthly_finance["net_cash_flow"] = (
    monthly_finance["income"]
    - monthly_finance["expense"]
)

# Check results
print("===== MONTHLY FINANCIAL DATASET =====")
print(monthly_finance.head(10))

print("\n===== DATASET SHAPE =====")
print(monthly_finance.shape)

print("\n===== UNIQUE ACCOUNTS =====")
print(monthly_finance["account_id"].nunique())

print("\n===== MONTHLY FINANCIAL SUMMARY =====")
print(
    monthly_finance[
        ["income", "expense", "net_cash_flow", "month_end_balance"]
    ].describe()
)

print("\n===== MISSING VALUES =====")
print(monthly_finance.isnull().sum())

===== MONTHLY FINANCIAL DATASET =====
   account_id year_month   income  expense  transaction_count  \
0           1    2015-03   100.00     0.00                  1   
1           1    2015-04  1629.82     0.00                  3   
2           1    2015-05   585.80     0.00                  3   
3           1    2015-06   377.96    20.00                  3   
4           1    2015-07   378.79   530.00                  3   
5           1    2015-08   377.82   751.46                  4   
6           1    2015-09   377.05   316.66                  5   
7           1    2015-10   376.95   536.66                  5   
8           1    2015-11   376.42   523.66                  6   
9           1    2015-12   375.80   460.66                  6   

   month_end_balance  net_cash_flow  
0             100.00         100.00  
1            1729.82        1629.82  
2            2315.62         585.80  
3            2673.58         357.96  
4            2522.37        -151.21  
5            2150.

## Monthly Timeline Validation

In [12]:
# STEP 9: Monthly Timeline Validation

monthly_finance = monthly_finance.sort_values(
    ["account_id", "year_month"]
).reset_index(drop=True)

# Check actual calendar gaps
monthly_finance["previous_month"] = (
    monthly_finance.groupby("account_id")["year_month"]
    .shift(1)
)

monthly_finance["month_gap"] = (
    (monthly_finance["year_month"].dt.year -
     monthly_finance["previous_month"].dt.year) * 12
    +
    (monthly_finance["year_month"].dt.month -
     monthly_finance["previous_month"].dt.month)
)

# Gaps greater than 1 month
missing_month_gaps = monthly_finance[
    monthly_finance["month_gap"] > 1
].copy()

missing_month_gaps["missing_months"] = (
    missing_month_gaps["month_gap"] - 1
)

print("===== MONTHLY TIMELINE REPORT =====")

print("Total Monthly Records:", len(monthly_finance))

print(
    "Accounts with Missing Months:",
    missing_month_gaps["account_id"].nunique()
)

print(
    "Total Missing Calendar Months:",
    int(missing_month_gaps["missing_months"].sum())
)

print("\n===== MONTH GAP DISTRIBUTION =====")
print(monthly_finance["month_gap"].value_counts().sort_index())

print("\n===== ACCOUNTS WITH LARGEST GAPS =====")

print(
    missing_month_gaps[
        ["account_id", "year_month", "month_gap", "missing_months"]
    ]
    .sort_values("missing_months", ascending=False)
    .head(10)
)

===== MONTHLY TIMELINE REPORT =====
Total Monthly Records: 184903
Accounts with Missing Months: 4493
Total Missing Calendar Months: 108729635

===== MONTH GAP DISTRIBUTION =====
month_gap
1        180271
2           118
3            11
4             4
5             2
          ...  
24225        83
24226        83
24227        78
24228        62
24229        64
Name: count, Length: 68, dtype: int64

===== ACCOUNTS WITH LARGEST GAPS =====
        account_id year_month  month_gap  missing_months
46195         1171    2017-12      24229           24228
46653         1186    2017-12      24229           24228
30568          777    2017-12      24229           24228
113970        2918    2017-12      24229           24228
112936        2892    2017-12      24229           24228
29785          758    2017-12      24229           24228
158964        4332    2017-12      24229           24228
131784        3385    2017-12      24229           24228
5272           134    2017-12      24229     

## Corrected Code

In [13]:
# STEP 9: FIXED MONTHLY TIMELINE VALIDATION

monthly_finance = monthly_finance.sort_values(
    ["account_id", "year_month"]
).reset_index(drop=True)

# Convert PeriodIndex to numeric month index
monthly_finance["month_number"] = (
    monthly_finance["year_month"].dt.year * 12
    + monthly_finance["year_month"].dt.month
)

# Calculate actual month gaps
monthly_finance["month_gap"] = (
    monthly_finance.groupby("account_id")["month_number"]
    .diff()
)

# First month of each account has no previous month
monthly_finance["month_gap"] = (
    monthly_finance["month_gap"].fillna(0).astype(int)
)

# Identify missing months
missing_month_gaps = monthly_finance[
    monthly_finance["month_gap"] > 1
].copy()

missing_month_gaps["missing_months"] = (
    missing_month_gaps["month_gap"] - 1
)

print("===== CORRECTED MONTHLY TIMELINE REPORT =====")

print("Total Monthly Records:", len(monthly_finance))

print(
    "Accounts with Missing Months:",
    missing_month_gaps["account_id"].nunique()
)

print(
    "Total Missing Calendar Months:",
    missing_month_gaps["missing_months"].sum()
)

print("\n===== MONTH GAP DISTRIBUTION =====")
print(
    monthly_finance["month_gap"]
    .value_counts()
    .sort_index()
)

print("\n===== LARGEST GAPS =====")
print(
    missing_month_gaps[
        ["account_id", "year_month", "month_gap", "missing_months"]
    ]
    .sort_values("missing_months", ascending=False)
    .head(10)
)

===== CORRECTED MONTHLY TIMELINE REPORT =====
Total Monthly Records: 184903
Accounts with Missing Months: 96
Total Missing Calendar Months: 185

===== MONTH GAP DISTRIBUTION =====
month_gap
0      4493
1    180271
2       118
3        11
4         4
5         2
6         1
7         2
9         1
Name: count, dtype: int64

===== LARGEST GAPS =====
        account_id year_month  month_gap  missing_months
9297           236    2018-12          9               8
148523        3823    2017-03          7               6
77802         1974    2018-05          7               6
79687         2021    2018-07          6               5
62244         1589    2018-08          5               4
80064         2029    2016-09          5               4
89732         2300    2018-05          4               3
44946         1138    2018-07          4               3
31376          799    2016-08          4               3
17329          438    2018-11          4               3


## Create Complete Monthly Timeline

In [14]:
# STEP 10: COMPLETE MONTHLY TIMELINE

# Preserve original aggregated data
monthly_original = monthly_finance[
    [
        "account_id",
        "year_month",
        "income",
        "expense",
        "net_cash_flow",
        "transaction_count",
        "month_end_balance"
    ]
].copy()

# Generate full monthly timeline per account
complete_timeline = []

for account_id, group in monthly_original.groupby("account_id"):

    months = pd.period_range(
        start=group["year_month"].min(),
        end=group["year_month"].max(),
        freq="M"
    )

    account_timeline = pd.DataFrame({
        "account_id": account_id,
        "year_month": months
    })

    complete_timeline.append(account_timeline)

complete_timeline = pd.concat(
    complete_timeline,
    ignore_index=True
)

# Merge actual financial data
monthly_complete = complete_timeline.merge(
    monthly_original,
    on=["account_id", "year_month"],
    how="left",
    validate="one_to_one"
)

# Flag missing months
monthly_complete["missing_month_flag"] = (
    monthly_complete["transaction_count"].isna()
).astype(int)

# Validate results
print("===== COMPLETE MONTHLY TIMELINE =====")
print("Original Records:", len(monthly_original))
print("Completed Records:", len(monthly_complete))

print(
    "Missing Months Added:",
    monthly_complete["missing_month_flag"].sum()
)

print(
    "Unique Accounts:",
    monthly_complete["account_id"].nunique()
)

print("\n===== MISSING MONTH SAMPLES =====")
print(
    monthly_complete[
        monthly_complete["missing_month_flag"] == 1
    ].head(10)
)

print("\n===== FINANCIAL DATA VALIDATION =====")
print("Original Income:", monthly_original["income"].sum())
print("Completed Income:", monthly_complete["income"].sum())

print("Original Expense:", monthly_original["expense"].sum())
print("Completed Expense:", monthly_complete["expense"].sum())

===== COMPLETE MONTHLY TIMELINE =====
Original Records: 184903
Completed Records: 185088
Missing Months Added: 185
Unique Accounts: 4493

===== MISSING MONTH SAMPLES =====
      account_id year_month  income  expense  net_cash_flow  \
621           18    2013-11     NaN      NaN            NaN   
4063         103    2017-01     NaN      NaN            NaN   
5444         140    2017-02     NaN      NaN            NaN   
6475         164    2017-11     NaN      NaN            NaN   
6477         164    2018-01     NaN      NaN            NaN   
9302         236    2018-04     NaN      NaN            NaN   
9303         236    2018-05     NaN      NaN            NaN   
9304         236    2018-06     NaN      NaN            NaN   
9305         236    2018-07     NaN      NaN            NaN   
9306         236    2018-08     NaN      NaN            NaN   

      transaction_count  month_end_balance  missing_month_flag  
621                 NaN                NaN                   1  
4063

## Time-Series Feature Engineering

In [15]:
# STEP 11: TIME-SERIES FEATURE ENGINEERING

ml_df = monthly_complete.copy()

ml_df = ml_df.sort_values(
    ["account_id", "year_month"]
).reset_index(drop=True)

# Calendar features
ml_df["month"] = ml_df["year_month"].dt.month
ml_df["year"] = ml_df["year_month"].dt.year
ml_df["quarter"] = ml_df["year_month"].dt.quarter

# Financial columns
financial_cols = [
    "income",
    "expense",
    "net_cash_flow",
    "month_end_balance"
]

# Lag features
for col in financial_cols:
    for lag in [1, 2, 3, 6, 12]:
        ml_df[f"{col}_lag_{lag}"] = (
            ml_df.groupby("account_id")[col].shift(lag)
        )

# Rolling features: shift before rolling
for col in ["income", "expense", "net_cash_flow"]:

    for window in [3, 6]:

        ml_df[f"{col}_rolling_{window}"] = (
            ml_df.groupby("account_id")[col]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=window,
                    min_periods=window
                ).mean()
            )
        )

# Check results
print("===== FEATURE ENGINEERING REPORT =====")

print("Dataset Shape:", ml_df.shape)

print("\nTotal Features:", len(ml_df.columns))

print("\nFeature Names:")
print(ml_df.columns.tolist())

print("\n===== FEATURE SAMPLE =====")
print(
    ml_df[
        [
            "account_id",
            "year_month",
            "income",
            "income_lag_1",
            "income_lag_3",
            "income_rolling_3",
            "expense",
            "expense_lag_1",
            "expense_rolling_3"
        ]
    ].head(15)
)

print("\n===== MISSING FEATURES =====")
print(
    ml_df.isnull().sum().sort_values(ascending=False).head(15)
)

===== FEATURE ENGINEERING REPORT =====
Dataset Shape: (185088, 37)

Total Features: 37

Feature Names:
['account_id', 'year_month', 'income', 'expense', 'net_cash_flow', 'transaction_count', 'month_end_balance', 'missing_month_flag', 'month', 'year', 'quarter', 'income_lag_1', 'income_lag_2', 'income_lag_3', 'income_lag_6', 'income_lag_12', 'expense_lag_1', 'expense_lag_2', 'expense_lag_3', 'expense_lag_6', 'expense_lag_12', 'net_cash_flow_lag_1', 'net_cash_flow_lag_2', 'net_cash_flow_lag_3', 'net_cash_flow_lag_6', 'net_cash_flow_lag_12', 'month_end_balance_lag_1', 'month_end_balance_lag_2', 'month_end_balance_lag_3', 'month_end_balance_lag_6', 'month_end_balance_lag_12', 'income_rolling_3', 'income_rolling_6', 'expense_rolling_3', 'expense_rolling_6', 'net_cash_flow_rolling_3', 'net_cash_flow_rolling_6']

===== FEATURE SAMPLE =====
    account_id year_month   income  income_lag_1  income_lag_3  \
0            1    2015-03   100.00           NaN           NaN   
1            1    2015-

## Next-Month Forecasting Targets

In [16]:
# STEP 12: NEXT-MONTH FORECASTING TARGETS

forecast_df = ml_df.copy()

forecast_df = forecast_df.sort_values(
    ["account_id", "year_month"]
).reset_index(drop=True)

# Target columns
target_mapping = {
    "income": "target_income",
    "expense": "target_expense",
    "net_cash_flow": "target_net_cash_flow",
    "month_end_balance": "target_balance"
}

# Next-month target creation
for col, target in target_mapping.items():

    forecast_df[target] = (
        forecast_df.groupby("account_id")[col]
        .shift(-1)
    )

# Forecast month
forecast_df["forecast_month"] = (
    forecast_df["year_month"] + 1
)

# Check results
print("===== FORECASTING TARGET REPORT =====")

print("Dataset Shape:", forecast_df.shape)

print("\n===== TARGET COLUMNS =====")
print(list(target_mapping.values()))

print("\n===== TARGET SAMPLE =====")
print(
    forecast_df[
        [
            "account_id",
            "year_month",
            "forecast_month",
            "income",
            "target_income",
            "expense",
            "target_expense",
            "target_net_cash_flow",
            "target_balance"
        ]
    ].head(15)
)

print("\n===== MISSING TARGET VALUES =====")

target_cols = list(target_mapping.values())

print(
    forecast_df[target_cols].isnull().sum()
)

print("\n===== VALID TARGET ROWS =====")

valid_targets = forecast_df.dropna(
    subset=target_cols
)

print("Total Rows:", len(forecast_df))
print("Valid Target Rows:", len(valid_targets))
print("Rows Without Complete Targets:",
      len(forecast_df) - len(valid_targets))

===== FORECASTING TARGET REPORT =====
Dataset Shape: (185088, 42)

===== TARGET COLUMNS =====
['target_income', 'target_expense', 'target_net_cash_flow', 'target_balance']

===== TARGET SAMPLE =====
    account_id year_month forecast_month   income  target_income  expense  \
0            1    2015-03        2015-04   100.00        1629.82     0.00   
1            1    2015-04        2015-05  1629.82         585.80     0.00   
2            1    2015-05        2015-06   585.80         377.96     0.00   
3            1    2015-06        2015-07   377.96         378.79    20.00   
4            1    2015-07        2015-08   378.79         377.82   530.00   
5            1    2015-08        2015-09   377.82         377.05   751.46   
6            1    2015-09        2015-10   377.05         376.95   316.66   
7            1    2015-10        2015-11   376.95         376.42   536.66   
8            1    2015-11        2015-12   376.42         375.80   523.66   
9            1    2015-12      

## Final ML Feature Selection

In [17]:
# STEP 13: ML FEATURE SELECTION

# Keep only rows with valid targets
model_df = forecast_df.dropna(
    subset=[
        "target_income",
        "target_expense",
        "target_net_cash_flow",
        "target_balance"
    ]
).copy()

# Future month seasonality
model_df["forecast_month_number"] = (
    model_df["forecast_month"].dt.month
)

model_df["forecast_quarter"] = (
    model_df["forecast_month"].dt.quarter
)

# Current month financial features
current_features = [
    "income",
    "expense",
    "net_cash_flow",
    "transaction_count",
    "month_end_balance",
    "missing_month_flag"
]

# Historical lag and rolling features
historical_features = [
    col for col in model_df.columns
    if "_lag_" in col or "_rolling_" in col
]

# Calendar features
calendar_features = [
    "month",
    "quarter",
    "forecast_month_number",
    "forecast_quarter"
]

# Final feature list
feature_cols = (
    current_features
    + historical_features
    + calendar_features
)

# Training targets
target_cols = [
    "target_income",
    "target_expense",
    "target_net_cash_flow",
    "target_balance"
]

# Input and target
X = model_df[feature_cols]
y = model_df[target_cols]

print("===== FINAL ML DATASET =====")

print("Total Rows:", len(model_df))
print("Total Input Features:", len(feature_cols))
print("Total Targets:", len(target_cols))

print("\n===== INPUT FEATURES =====")
print(feature_cols)

print("\n===== TARGET COLUMNS =====")
print(target_cols)

print("\n===== FEATURE MISSING VALUES =====")
print(
    X.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\n===== FORECAST PERIOD =====")
print("Start:", model_df["forecast_month"].min())
print("End:", model_df["forecast_month"].max())

print("\n===== DUPLICATE ACCOUNT-MONTH ROWS =====")
print(
    model_df.duplicated(
        subset=["account_id", "forecast_month"]
    ).sum()
)

===== FINAL ML DATASET =====
Total Rows: 180410
Total Input Features: 36
Total Targets: 4

===== INPUT FEATURES =====
['income', 'expense', 'net_cash_flow', 'transaction_count', 'month_end_balance', 'missing_month_flag', 'income_lag_1', 'income_lag_2', 'income_lag_3', 'income_lag_6', 'income_lag_12', 'expense_lag_1', 'expense_lag_2', 'expense_lag_3', 'expense_lag_6', 'expense_lag_12', 'net_cash_flow_lag_1', 'net_cash_flow_lag_2', 'net_cash_flow_lag_3', 'net_cash_flow_lag_6', 'net_cash_flow_lag_12', 'month_end_balance_lag_1', 'month_end_balance_lag_2', 'month_end_balance_lag_3', 'month_end_balance_lag_6', 'month_end_balance_lag_12', 'income_rolling_3', 'income_rolling_6', 'expense_rolling_3', 'expense_rolling_6', 'net_cash_flow_rolling_3', 'net_cash_flow_rolling_6', 'month', 'quarter', 'forecast_month_number', 'forecast_quarter']

===== TARGET COLUMNS =====
['target_income', 'target_expense', 'target_net_cash_flow', 'target_balance']

===== FEATURE MISSING VALUES =====
net_cash_flow_lag

## Chronological Train–Validation–Test Split

In [18]:
# STEP 14: CHRONOLOGICAL DATA SPLIT

# Make a separate copy
split_df = model_df.copy()

# Convert forecast month to timestamp
split_df["forecast_date"] = (
    split_df["forecast_month"].dt.to_timestamp()
)

# Chronological split
train_df = split_df[
    split_df["forecast_date"] < "2017-01-01"
].copy()

val_df = split_df[
    (split_df["forecast_date"] >= "2017-01-01") &
    (split_df["forecast_date"] < "2018-01-01")
].copy()

test_df = split_df[
    (split_df["forecast_date"] >= "2018-01-01") &
    (split_df["forecast_date"] <= "2018-12-31")
].copy()

# Input features
X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

# Prediction targets
y_train = train_df[target_cols]
y_val = val_df[target_cols]
y_test = test_df[target_cols]

print("===== CHRONOLOGICAL SPLIT REPORT =====")

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\n===== TARGET SHAPES =====")

print("Training:", y_train.shape)
print("Validation:", y_val.shape)
print("Testing:", y_test.shape)

print("\n===== DATE RANGES =====")

for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        "| Start:", data["forecast_month"].min(),
        "| End:", data["forecast_month"].max(),
        "| Accounts:", data["account_id"].nunique()
    )

print("\n===== DATASET PERCENTAGES =====")

total = len(split_df)

print("Training:", round(len(train_df) / total * 100, 2), "%")
print("Validation:", round(len(val_df) / total * 100, 2), "%")
print("Testing:", round(len(test_df) / total * 100, 2), "%")

print("\n===== CHECK FOR EMPTY SPLITS =====")

print("Train Empty:", train_df.empty)
print("Validation Empty:", val_df.empty)
print("Test Empty:", test_df.empty)

===== CHRONOLOGICAL SPLIT REPORT =====
Training: (78753, 36)
Validation: (48006, 36)
Testing: (53651, 36)

===== TARGET SHAPES =====
Training: (78753, 4)
Validation: (48006, 4)
Testing: (53651, 4)

===== DATE RANGES =====
Train | Start: 2013-02 | End: 2016-12 | Accounts: 3469
Validation | Start: 2017-01 | End: 2017-12 | Accounts: 4426
Test | Start: 2018-01 | End: 2018-12 | Accounts: 4485

===== DATASET PERCENTAGES =====
Training: 43.65 %
Validation: 26.61 %
Testing: 29.74 %

===== CHECK FOR EMPTY SPLITS =====
Train Empty: False
Validation Empty: False
Test Empty: False


## Last Month's Value = Next Month's Prediction

In [19]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error

# STEP 15: BASELINE FORECASTING

def evaluate_forecast(actual, predicted):

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    return mae, rmse


baseline_results = []

for name, data in [
    ("Validation", val_df),
    ("Test", test_df)
]:

    # Current month values predict next month
    baseline_predictions = {
        "Income": (
            data["target_income"],
            data["income"]
        ),
        "Expense": (
            data["target_expense"],
            data["expense"]
        ),
        "Net Cash Flow": (
            data["target_net_cash_flow"],
            data["net_cash_flow"]
        ),
        "Balance": (
            data["target_balance"],
            data["month_end_balance"]
        )
    }

    for target_name, (actual, predicted) in baseline_predictions.items():

        # Skip rows with missing baseline inputs
        valid = actual.notna() & predicted.notna()

        if valid.sum() == 0:
            continue

        mae, rmse = evaluate_forecast(
            actual[valid],
            predicted[valid]
        )

        baseline_results.append({
            "Dataset": name,
            "Target": target_name,
            "Rows": int(valid.sum()),
            "MAE": round(mae, 2),
            "RMSE": round(rmse, 2)
        })


baseline_report = pd.DataFrame(baseline_results)

print("===== BASELINE FORECASTING REPORT =====")
print(baseline_report.to_string(index=False))

===== BASELINE FORECASTING REPORT =====
   Dataset        Target  Rows     MAE    RMSE
Validation        Income 47967  811.79 2044.00
Validation       Expense 47967  976.13 1894.70
Validation Net Cash Flow 47967 1092.38 1988.49
Validation       Balance 47967  733.00 1295.91
      Test        Income 53615  804.22 2050.63
      Test       Expense 53615 1058.01 2007.27
      Test Net Cash Flow 53615 1136.09 2043.60
      Test       Balance 53615  744.73 1308.78


## Final Data Preparation

In [23]:
# Remove unusable current-month observations

def prepare_data(data):

    df = data.copy()

    current_cols = [
        "income",
        "expense",
        "net_cash_flow",
        "transaction_count",
        "month_end_balance"
    ]

    valid = (
        (df["missing_month_flag"] == 0)
        & df[current_cols].notna().all(axis=1)
    )

    df = df.loc[valid].copy()

    # Seasonal features for the forecast month
    m = df["forecast_month"].dt.month

    df["forecast_month_sin"] = np.sin(
        2 * np.pi * m / 12
    )

    df["forecast_month_cos"] = np.cos(
        2 * np.pi * m / 12
    )

    return df.sort_values(
        ["forecast_month", "account_id"]
    ).reset_index(drop=True)


train_ml = prepare_data(train_df)
val_ml = prepare_data(val_df)
test_ml = prepare_data(test_df)

# Missing-month flag is always 0 after filtering
ml_features = [
    col for col in feature_cols
    if col != "missing_month_flag"
]

ml_features += [
    "forecast_month_sin",
    "forecast_month_cos"
]

# Create feature matrices
X_train = train_ml[ml_features]
X_val = val_ml[ml_features]
X_test = test_ml[ml_features]

print("===== FINAL DATA PREPARATION =====")

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nInput Features:", len(ml_features))

print("\nRemaining Missing Feature Values:")
print(X_train.isnull().sum().sum())

print("\nTarget Availability:")
for name, target in TARGETS.items():
    print(name, train_ml[target].notna().sum())

===== FINAL DATA PREPARATION =====
Train: (78689, 37)
Validation: (47967, 37)
Test: (53615, 37)

Input Features: 37

Remaining Missing Feature Values:
383053

Target Availability:
Income 78689
Expense 78689
Balance 78689


## Evaluation Functions and Fair Baseline

In [24]:
def evaluate_model(actual, predicted):

    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(actual, predicted)
    }


def fix_prediction(prediction, target_name):

    prediction = np.asarray(prediction).ravel()

    # Income and expense cannot be negative
    if target_name in ["Income", "Expense"]:
        prediction = np.maximum(prediction, 0)

    return prediction


# Baseline: current month predicts next month
baseline_columns = {
    "Income": "income",
    "Expense": "expense",
    "Balance": "month_end_balance"
}

baseline_validation = {}

for name, target in TARGETS.items():

    scores = evaluate_model(
        val_ml[target],
        val_ml[baseline_columns[name]]
    )

    baseline_validation[name] = scores


print("===== FAIR VALIDATION BASELINE =====")

for name, scores in baseline_validation.items():

    print(
        name,
        "| MAE:", round(scores["MAE"], 2),
        "| RMSE:", round(scores["RMSE"], 2)
    )

===== FAIR VALIDATION BASELINE =====
Income | MAE: 811.79 | RMSE: 2044.0
Expense | MAE: 976.13 | RMSE: 1894.7
Balance | MAE: 733.0 | RMSE: 1295.91


In [25]:
def suggest_params(trial, model_name):

    if model_name == "LightGBM":

        return {
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.015, 0.15, log=True
            ),
            "num_leaves": trial.suggest_int(
                "num_leaves", 15, 63
            ),
            "max_depth": trial.suggest_int(
                "max_depth", 4, 10
            ),
            "min_child_samples": trial.suggest_int(
                "min_child_samples", 15, 100
            ),
            "subsample": trial.suggest_float(
                "subsample", 0.65, 1.0
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.65, 1.0
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha", 1e-4, 10.0, log=True
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda", 1e-3, 20.0, log=True
            )
        }

    elif model_name == "XGBoost":

        return {
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.015, 0.15, log=True
            ),
            "max_depth": trial.suggest_int(
                "max_depth", 3, 9
            ),
            "min_child_weight": trial.suggest_int(
                "min_child_weight", 1, 15
            ),
            "subsample": trial.suggest_float(
                "subsample", 0.65, 1.0
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.65, 1.0
            ),
            "gamma": trial.suggest_float(
                "gamma", 0.0, 5.0
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha", 1e-4, 10.0, log=True
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda", 1e-3, 20.0, log=True
            )
        }

    elif model_name == "CatBoost":

        return {
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.015, 0.15, log=True
            ),
            "depth": trial.suggest_int(
                "depth", 4, 8
            ),
            "l2_leaf_reg": trial.suggest_float(
                "l2_leaf_reg", 0.1, 20.0, log=True
            ),
            "random_strength": trial.suggest_float(
                "random_strength", 0.0, 3.0
            )
        }

    elif model_name == "ElasticNet":

        return {
            "alpha": trial.suggest_float(
                "alpha", 0.001, 1000.0, log=True
            ),
            "l1_ratio": trial.suggest_float(
                "l1_ratio", 0.05, 0.95
            )
        }

    else:
        raise ValueError("Unknown model: " + model_name)


print("Hyperparameter search spaces created.")

Hyperparameter search spaces created.


## Create Model Factory

In [26]:
def build_model(
    model_name,
    params,
    n_estimators=None,
    early_stopping=False
):

    if model_name == "LightGBM":

        return LGBMRegressor(
            **params,
            n_estimators=n_estimators or 1400,
            objective="regression",
            metric="mae",
            subsample_freq=1,
            random_state=RANDOM_STATE,
            n_jobs=CPU_THREADS,
            verbosity=-1
        )

    elif model_name == "XGBoost":

        return XGBRegressor(
            **params,
            n_estimators=n_estimators or 1400,
            objective="reg:squarederror",
            tree_method="hist",
            eval_metric="mae",
            early_stopping_rounds=70 if early_stopping else None,
            random_state=RANDOM_STATE,
            n_jobs=CPU_THREADS
        )

    elif model_name == "CatBoost":

        return CatBoostRegressor(
            **params,
            iterations=n_estimators or 1400,
            loss_function="RMSE",
            eval_metric="MAE",
            random_seed=RANDOM_STATE,
            thread_count=CPU_THREADS,
            allow_writing_files=False,
            verbose=False
        )

    elif model_name == "ElasticNet":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                ElasticNet(
                    **params,
                    max_iter=5000,
                    tol=1e-3,
                    selection="random",
                    random_state=RANDOM_STATE
                )
            )
        ])

    else:
        raise ValueError("Unknown model: " + model_name)


print("All four model architectures ready.")

All four model architectures ready.


##  Optuna Optimization Function

In [28]:
def optimize_model(model_name, target_name, target_col):

    y_tr = train_ml[target_col].to_numpy()
    y_va = val_ml[target_col].to_numpy()

    def objective(trial):

        params = suggest_params(trial, model_name)

        model = build_model(
            model_name,
            params,
            early_stopping=True
        )

        if model_name == "LightGBM":

            model.fit(
                X_train,
                y_tr,
                eval_set=[(X_val, y_va)],
                callbacks=[
                    lgb.early_stopping(
                        70,
                        verbose=False
                    )
                ]
            )

            best_iterations = (
                model.best_iteration_ or model.n_estimators
            )

        elif model_name == "XGBoost":

            model.fit(
                X_train,
                y_tr,
                eval_set=[(X_val, y_va)],
                verbose=False
            )

            best_iterations = model.best_iteration + 1

        elif model_name == "CatBoost":

            model.fit(
                X_train,
                y_tr,
                eval_set=(X_val, y_va),
                early_stopping_rounds=70,
                use_best_model=True,
                verbose=False
            )

            best_iterations = model.tree_count_

        else:

            model.fit(X_train, y_tr)

            best_iterations = None

        predictions = fix_prediction(
            model.predict(X_val),
            target_name
        )

        mae = mean_absolute_error(
            y_va,
            predictions
        )

        if best_iterations is not None:
            trial.set_user_attr(
                "best_iterations",
                int(best_iterations)
            )

        return mae

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE,
            n_startup_trials=5
        )
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS[model_name],
        gc_after_trial=True
    )

    return study


print("Optuna optimization function ready.")

Optuna optimization function ready.


## Run Hyperparameter Tuning for All Models

In [29]:
# MAIN HYPERPARAMETER TUNING

tuning_results = {}
tuning_summary = []

start_time = time.time()

for target_name, target_col in TARGETS.items():

    tuning_results[target_name] = {}

    print(f"\n========== {target_name.upper()} ==========")

    for model_name in MODEL_NAMES:

        print(f"\nTuning {model_name} for {target_name}...")

        model_start = time.time()

        study = optimize_model(
            model_name,
            target_name,
            target_col
        )

        best_iterations = study.best_trial.user_attrs.get(
            "best_iterations"
        )

        tuning_results[target_name][model_name] = {
            "params": study.best_params,
            "best_iterations": best_iterations,
            "validation_mae": study.best_value
        }

        elapsed = (time.time() - model_start) / 60

        tuning_summary.append({
            "Target": target_name,
            "Model": model_name,
            "Best MAE": round(study.best_value, 2),
            "Best Trees": best_iterations,
            "Time (min)": round(elapsed, 2)
        })

        print(
            f"Completed: {model_name}"
            f" | Best MAE: {study.best_value:.2f}"
            f" | Time: {elapsed:.2f} min"
        )


tuning_report = pd.DataFrame(tuning_summary)

print("\n===== OPTUNA TUNING REPORT =====")
print(tuning_report.to_string(index=False))

print("\nTotal Training Time (Minutes):")
print(round((time.time() - start_time) / 60, 2))


========== INCOME ==========

Tuning LightGBM for Income...


c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

Completed: LightGBM | Best MAE: 581.95 | Time: 1.04 min

Tuning CatBoost for Income...
Completed: CatBoost | Best MAE: 587.38 | Time: 4.61 min

Tuning XGBoost for Income...
Completed: XGBoost | Best MAE: 576.09 | Time: 4.04 min

Tuning ElasticNet for Income...


c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.774e+10, tolerance: 3.062e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.670e+10, tolerance: 3.062e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sca

Completed: ElasticNet | Best MAE: 707.30 | Time: 4.70 min

========== EXPENSE ==========

Tuning LightGBM for Expense...


...

c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

Completed: LightGBM | Best MAE: 593.52 | Time: 2.74 min

Tuning CatBoost for Balance...
Completed: CatBoost | Best MAE: 590.78 | Time: 5.61 min

Tuning XGBoost for Balance...
Completed: XGBoost | Best MAE: 591.06 | Time: 9.74 min

Tuning ElasticNet for Balance...


c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.000e+10, tolerance: 2.441e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.483e+10, tolerance: 2.441e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sca

Completed: ElasticNet | Best MAE: 666.35 | Time: 2.84 min

===== OPTUNA TUNING REPORT =====
 Target      Model  Best MAE  Best Trees  Time (min)
 Income   LightGBM    581.95       435.0        1.04
 Income   CatBoost    587.38      1397.0        4.61
 Income    XGBoost    576.09       530.0        4.04
 Income ElasticNet    707.30         NaN        4.70
Expense   LightGBM    605.23       619.0        1.76
Expense   CatBoost    603.56       610.0        3.06
Expense    XGBoost    600.03       518.0        5.17
Expense ElasticNet    708.37         NaN        2.57
Balance   LightGBM    593.52      1335.0        2.74
Balance   CatBoost    590.78      1400.0        5.61
Balance    XGBoost    591.06      1029.0        9.74
Balance ElasticNet    666.35         NaN        2.84

Total Training Time (Minutes):
47.87


c:\Users\INTEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.879e+10, tolerance: 2.441e+08
  model = cd_fast.enet_coordinate_descent(
